### 01.Pytorch Setup and DataLoader Creation
Explanation: Deep learning models train in "batches" rather than looking at the entire dataset at once. We convert our numpy arrays into PyTorch Tensors and wrap them in DataLoaders. We also calculate class weights to penalize the model if it tries to blindly guess the majority class (just like we did with XGBoost).

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
import copy

print("Block 1: Loading Data and Creating DataLoaders...")

# 1. Load the preprocessed 3D data
data = np.load("preprocessed_data.npz")

X_train_3d = data['X_train_3d']
X_val_3d = data['X_val_3d']
X_test_3d = data['X_test_3d']

y_train_cls = data['y_train_cls_3d']
y_val_cls = data['y_val_cls_3d']
y_test_cls = data['y_test_cls_3d']

# 2. Convert to PyTorch Tensors
X_train_tensor = torch.tensor(X_train_3d, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_cls, dtype=torch.long)

X_val_tensor = torch.tensor(X_val_3d, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_cls, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_3d, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_cls, dtype=torch.long)

# 3. Create DataLoaders
BATCH_SIZE = 64

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 4. Calculate Class Weights for Loss Function
# This forces the network to care equally about Up (1) and Down (0) days
class_counts = np.bincount(y_train_cls)
total_samples = len(y_train_cls)
class_weights = total_samples / (2.0 * class_counts)
weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print(f"DataLoaders Ready.")
print(f"Input Shape: {X_train_tensor.shape} -> (Samples, Lookback, Features)")
print(f"Class Weights [Down, Up]: {class_weights}")

Block 1: Loading Data and Creating DataLoaders...
DataLoaders Ready.
Input Shape: torch.Size([1849, 30, 47]) -> (Samples, Lookback, Features)
Class Weights [Down, Up]: [1.07375145 0.93572874]


### 02.The Neural Network Architecture
Explanation: This is the core engine. The nn.LSTM processes the sequence. The Attention step calculates a score for each of the 30 days, squashes those scores using a Softmax function, and multiplies them by the LSTM output. This creates a "context vector" that summarizes the most important events of the past month.

In [5]:
print("\nBlock 2: Defining the Attention-LSTM Architecture...")

class AttentionLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim, dropout=0.3):
        super(AttentionLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, 
                            num_layers=num_layers, batch_first=True, 
                            bidirectional=True, dropout=dropout if num_layers > 1 else 0)
        self.attention_linear = nn.Linear(hidden_dim * 2, 1)
        self.fc1 = nn.Linear(hidden_dim * 2, 32)
        self.fc2 = nn.Linear(32, output_dim)
        
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_weights = torch.softmax(self.attention_linear(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return self.fc2(torch.relu(self.fc1(context)))

# Initialize the model
INPUT_DIM = X_train_tensor.shape[2] # 47 features
HIDDEN_DIM = 64
NUM_LAYERS = 2
OUTPUT_DIM = 2 

model = AttentionLSTM(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS, OUTPUT_DIM)
print("Model initialized successfully!")


Block 2: Defining the Attention-LSTM Architecture...
Model initialized successfully!


In [6]:
print("\nBlock 3: Training the Model with Gradient Clipping & Class Weights...")

# --- HYPERPARAMETER TWEAKS ---
EPOCHS = 100
LEARNING_RATE = 0.0005 
PATIENCE = 15

# 1. THE CURE FOR MODE COLLAPSE: Put the weights_tensor BACK!
# This mathematically forces the LSTM to care about downward (0) days.
criterion = nn.CrossEntropyLoss(weight=weights_tensor) 

# 2. Increased weight_decay slightly for stronger L2 Regularization
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4) 

best_val_loss = float('inf')
patience_counter = 0
best_model_weights = copy.deepcopy(model.state_dict())

for epoch in range(EPOCHS):
    # --- TRAINING PHASE ---
    model.train()
    train_loss = 0.0
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        
        # 3. GRADIENT CLIPPING (The secret to stable LSTMs in finance)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
    
    train_loss = train_loss / len(train_loader.dataset)
    
    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, targets in val_loader:
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * inputs.size(0)
            
    val_loss = val_loss / len(val_loader.dataset)
    
    # Early Stopping Logic
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_weights = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:03d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Patience: {patience_counter}/{PATIENCE}")
        
    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch+1}. Restoring best model weights.")
        break

# Restore the best model for testing
model.load_state_dict(best_model_weights)
print("Training Complete.")


Block 3: Training the Model with Gradient Clipping & Class Weights...
Epoch 001/100 | Train Loss: nan | Val Loss: nan | Patience: 1/15
Epoch 005/100 | Train Loss: nan | Val Loss: nan | Patience: 5/15
Epoch 010/100 | Train Loss: nan | Val Loss: nan | Patience: 10/15
Epoch 015/100 | Train Loss: nan | Val Loss: nan | Patience: 15/15

Early stopping triggered at epoch 15. Restoring best model weights.
Training Complete.


In [7]:
print("Cleaning NaNs from PyTorch Tensors...")

# Check how many NaNs exist before cleaning
print(f"NaNs in Train before: {torch.isnan(X_train_tensor).sum().item()}")

# Replace all NaNs with 0.0 (which equals the Mean because of StandardScaler)
X_train_tensor = torch.nan_to_num(X_train_tensor, nan=0.0)
X_val_tensor   = torch.nan_to_num(X_val_tensor, nan=0.0)
X_test_tensor  = torch.nan_to_num(X_test_tensor, nan=0.0)

print(f"NaNs in Train after: {torch.isnan(X_train_tensor).sum().item()}")

# Recreate the DataLoaders with the cleaned tensors
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("DataLoaders Rebuilt. Safe to train!")

Cleaning NaNs from PyTorch Tensors...
NaNs in Train before: 17561
NaNs in Train after: 0
DataLoaders Rebuilt. Safe to train!


### 03. The training Loop with Early Stopping
Explanation: Deep neural networks will memorize the training data if left unchecked. We use a training loop that evaluates the model on the Validation set after every epoch. If the validation loss stops improving for 15 consecutive epochs (patience=15), training stops immediately, and we restore the weights from the absolute best epoch.

In [8]:
print("\nBlock 3: Training the Model with Gradient Clipping (Updated)...")

# --- HYPERPARAMETER TWEAKS ---
EPOCHS = 100
LEARNING_RATE = 0.0005  # Lowered learning rate for stability
PATIENCE = 15

# 1. REMOVED the weight_tensor. Let the model learn naturally.
criterion = nn.CrossEntropyLoss() 

# 2. Increased weight_decay slightly for stronger L2 Regularization
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4) 

best_val_loss = float('inf')
patience_counter = 0
best_model_weights = copy.deepcopy(model.state_dict())

for epoch in range(EPOCHS):
    # --- TRAINING PHASE ---
    model.train()
    train_loss = 0.0
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        
        # 3. GRADIENT CLIPPING (The secret to stable LSTMs in finance)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
    
    train_loss = train_loss / len(train_loader.dataset)
    
    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, targets in val_loader:
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * inputs.size(0)
            
    val_loss = val_loss / len(val_loader.dataset)
    
    # Early Stopping Logic
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_weights = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:03d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Patience: {patience_counter}/{PATIENCE}")
        
    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch+1}. Restoring best model weights.")
        break

# Restore the best model for testing
model.load_state_dict(best_model_weights)
print("Training Complete.")


Block 3: Training the Model with Gradient Clipping (Updated)...
Epoch 001/100 | Train Loss: 0.6989 | Val Loss: 0.6895 | Patience: 0/15
Epoch 005/100 | Train Loss: 0.6857 | Val Loss: 0.7089 | Patience: 4/15
Epoch 010/100 | Train Loss: 0.6823 | Val Loss: 0.7297 | Patience: 9/15
Epoch 015/100 | Train Loss: 0.6771 | Val Loss: 0.7366 | Patience: 14/15

Early stopping triggered at epoch 16. Restoring best model weights.
Training Complete.


### 04.Evaluation on the Test Set
Explanation: Finally, we run the completely unseen X_test_tensor through the optimized network and generate the classification report to see how it compares against the XGBoost baseline.

In [9]:
print("\nBlock 4: Evaluating on the Test Set...")

model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for inputs, targets in test_loader:
        outputs = model(inputs)
        # Apply Softmax to get probabilities, then take the argmax to get the predicted class (0 or 1)
        probabilities = torch.softmax(outputs, dim=1)
        _, preds = torch.max(probabilities, 1)
        
        all_preds.extend(preds.numpy())
        all_targets.extend(targets.numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

# Calculate metrics
acc = accuracy_score(all_targets, all_preds)
print(f"\n--- Deep Learning Sequence Tower Evaluation ---")
print(f"Test Accuracy: {acc * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(all_targets, all_preds, zero_division=0))


Block 4: Evaluating on the Test Set...

--- Deep Learning Sequence Tower Evaluation ---
Test Accuracy: 53.08%

Classification Report:
              precision    recall  f1-score   support

           0       0.51      0.49      0.50       177
           1       0.55      0.57      0.56       196

    accuracy                           0.53       373
   macro avg       0.53      0.53      0.53       373
weighted avg       0.53      0.53      0.53       373



In [10]:
import os

print("\nBlock 5: Saving the Fixed LSTM Weights...")

# Ensure the saved_models directory exists
os.makedirs("saved_models", exist_ok=True)

# Save the PyTorch state dictionary
torch.save(model.state_dict(), "saved_models/lstm_weights.pth")

print("✅ Fixed LSTM weights saved successfully to 'saved_models/lstm_weights.pth'!")


Block 5: Saving the Fixed LSTM Weights...
✅ Fixed LSTM weights saved successfully to 'saved_models/lstm_weights.pth'!
